In [40]:
import pandas as pd
import numpy as np
from pathlib import Path


In [41]:
csv_path = Path(r"C:\Users\aynur\Downloads\archive (7)\Salary Data.csv")
df = pd.read_csv(csv_path)
df

,Age,Gender,Education Level,Job Title,Years of Experience,Salary
0,32.0,Male,Bachelor's,Software Engineer,5.0,90000.0
1,28.0,Female,Master's,Data Analyst,3.0,65000.0
2,45.0,Male,PhD,Senior Manager,15.0,150000.0
3,36.0,Female,Bachelor's,Sales Associate,7.0,60000.0
4,52.0,Male,Master's,Director,20.0,200000.0
...,...,...,...,...,...,...
370,35.0,Female,Bachelor's,Senior Marketing Analyst,8.0,85000.0
371,43.0,Male,Master's,Director of Operations,19.0,170000.0
372,29.0,Female,Bachelor's,Junior Project Manager,2.0,40000.0
373,34.0,Male,Bachelor's,Senior Operations Coordinator,7.0,90000.0


In [42]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 375 entries, 0 to 374
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Age                  373 non-null    float64
 1   Gender               373 non-null    object 
 2   Education Level      373 non-null    object 
 3   Job Title            373 non-null    object 
 4   Years of Experience  373 non-null    float64
 5   Salary               373 non-null    float64
dtypes: float64(3), object(3)
memory usage: 17.7+ KB


In [43]:
df = df[df['Salary'] > 1000]

In [44]:
df = df.dropna(subset=['Salary'])

In [45]:
X = df.drop('Salary', axis = 'columns')
y = df['Salary']

In [46]:
num_cols = X.select_dtypes(include = 'number').columns.to_list()
cat_cols = X.select_dtypes(include = 'object').columns.to_list()

In [47]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

num_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy = 'median')),
    ('scaling', StandardScaler())
])

cat_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy = 'most_frequent')),
    ('encoding', OneHotEncoder(sparse_output= False, handle_unknown= 'ignore'))
])

preprocessing = ColumnTransformer([
    ('NUM', num_pipeline, num_cols),
    ('CAT', cat_pipeline, cat_cols)
])



In [48]:
from pandas.core.common import random_state
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [49]:
from sklearn.linear_model import LassoCV

full_pipeline_cv = Pipeline([
    ('cleaning', preprocessing),
    ('model', LassoCV(cv=5, random_state=42))
])

full_pipeline_cv.fit(X_train, y_train)
print("Ən yaxşı alpha:", full_pipeline_cv.named_steps['model'].alpha_)

Ən yaxşı alpha: 87.81394341401767


In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.linear_model import Lasso
from sklearn.metrics import root_mean_squared_error
best_alpha = full_pipeline_cv.named_steps['model'].alpha_
print("Ən yaxşı alpha:", best_alpha)


full_pipeline_lasso = Pipeline([
    ('cleaning', preprocessing),
    ('model', Lasso(alpha=best_alpha))
])
full_pipeline_lasso.fit(X_train, y_train)


y_train_predict = full_pipeline_lasso.predict(X_train)
y_test_predict = full_pipeline_lasso.predict(X_test)


train_rmse = root_mean_squared_error(y_train, y_train_predict)
test_rmse = root_mean_squared_error(y_test, y_test_predict)

print("Lasso Train RMSE:", train_rmse)
print("Lasso Test RMSE:", test_rmse)

Ən yaxşı alpha: 87.81394341401767
Lasso Train RMSE: 11802.568091084491
Lasso Test RMSE: 14767.536212175406


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

full_pipeline_rf = Pipeline([
    ('cleaning', preprocessing),
    ('model', RandomForestRegressor(
        n_estimators=200,
        max_depth=15,        
        min_samples_leaf=3,  
        random_state=42
    ))
])

full_pipeline_rf.fit(X_train, y_train)

y_train_pred_rf = full_pipeline_rf.predict(X_train)
y_test_pred_rf = full_pipeline_rf.predict(X_test)

print("RF Train RMSE:", root_mean_squared_error(y_train, y_train_pred_rf))
print("RF Test RMSE:", root_mean_squared_error(y_test, y_test_pred_rf))

RF Train RMSE: 11586.827270364482
RF Test RMSE: 16410.479365081625


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [10, 15, 20, None],
    'model__min_samples_leaf': [2, 3, 5]
}

grid_search = GridSearchCV(
    full_pipeline_rf,
    param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1, 
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Ən yaxşı parametrlər:", grid_search.best_params_)
best_rf = grid_search.best_estimator_

print("RF Train RMSE:", root_mean_squared_error(y_train, best_rf.predict(X_train)))
print("RF Test RMSE:", root_mean_squared_error(y_test, best_rf.predict(X_test)))

Fitting 5 folds for each of 24 candidates, totalling 120 fits
Ən yaxşı parametrlər: {'model__max_depth': 20, 'model__min_samples_leaf': 2, 'model__n_estimators': 200}
RF Train RMSE: 9599.176495751615
RF Test RMSE: 16871.21585645231


In [53]:
from sklearn.ensemble import GradientBoostingRegressor

full_pipeline_gb = Pipeline([
    ('cleaning', preprocessing),
    ('model', GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        random_state=42
    ))
])

full_pipeline_gb.fit(X_train, y_train)

print("GB Train RMSE:", root_mean_squared_error(y_train, full_pipeline_gb.predict(X_train)))
print("GB Test RMSE:", root_mean_squared_error(y_test, full_pipeline_gb.predict(X_test)))

GB Train RMSE: 6458.075928588776
GB Test RMSE: 18098.23054365504


In [54]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    full_pipeline_gb,
    X_train, y_train,
    scoring='neg_root_mean_squared_error',
    cv=5
)

print("CV RMSE-ləri:", -scores)
print("Ortalama CV RMSE:", -scores.mean())
print("Std:", scores.std())

CV RMSE-ləri: [14557.23848285 11759.58676862 15193.54775695 16831.22293633
 14984.25701143]
Ortalama CV RMSE: 14665.17059123618
Std: 1644.8007797175176


In [55]:
print(df.shape)
print(y.describe())

(372, 6)
count       372.000000
mean     100846.774194
std       48023.137565
min       30000.000000
25%       55000.000000
50%       95000.000000
75%      140000.000000
max      250000.000000
Name: Salary, dtype: float64


In [56]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# plt.figure(figsize=(10, 6))
# sns.scatterplot(x=y_test, y=y_test_predict)
# plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--r', linewidth=2)
# plt.xlabel('Real Maaşlar')
# plt.ylabel('Proqnozlaşdırılan Maaşlar')
# plt.title('Real vs Proqnoz (Lasso)')
# plt.show()